# Computational Set: Data Analysis B — Multiple (Polynomial) Regression

### Calibration of a Thermistor

**Learning objectives**

- Extend linear regression to a **polynomial** model and extract every coefficient with its uncertainty.
- Build derived columns and judge a curved fit.
- Report parameters with correct units and significant figures.

---

#### Background

A **thermistor** is a ceramic semiconductor whose resistance changes steeply with temperature,
letting us measure temperature to ~0.001 °C — but the response is strongly **nonlinear**. The
inverse absolute temperature is well described by a *quadratic* function of the natural log of
the resistance (divided by 1 Ω, since logs must be of a unitless quantity):

$$
\frac{1}{T} = b + m_1\,\ln(R/\Omega) + m_2\,\bigl[\ln(R/\Omega)\bigr]^2. \tag{3}
$$

This is exactly linear regression with one extra term. In Excel you selected *two* x-columns
($\ln R$ and $(\ln R)^2$) in the Regression tool — "multiple regression." In Python it's a
one-line change: `numpy.polyfit` with degree **2** instead of 1.


> **New syntax in this set (building on Exercise A).** Two things to know:
> - `np.log(x)` is the *natural* logarithm (base $e$). It acts element-by-element on an array.
> - `np.polyfit(x, y, 2, cov=True)` now fits a **quadratic**, so it returns **three**
>   coefficients, still **highest power first**: `[m2, m1, b]`. Unpack them in that order.
>
> If you'd like a fuller Python refresher, see the ESCIP
> [“What is Python?” notebook](https://escip.io/notebooks/python/python-basics-fixed.html).

## The data

For each calibration point we have the thermistor resistance `R` (in Ω) and the bath
temperature `T_C` (in °C) measured with a reference thermometer.

Run the cell below to load the data — **you don't need to edit it.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

R   = np.array([1000, 1800, 3200, 5600, 10000, 18000, 32000, 56000, 100000], dtype=float)  # ohms
T_C = np.array([75.05, 58.84, 45.68, 34.48, 25.00, 16.74, 10.03, 4.57, -0.01])             # deg C

print(f"{len(R)} calibration points loaded.")

## Step 1 — Build the derived columns

Equation (3) treats $\ln(R/\Omega)$ as the predictor and $1/T$ (with $T$ in **kelvin**) as the
response. Build three arrays:

- `x`  — the natural log of the resistance,
- `T_K` — the absolute temperature (add 273.15 to the Celsius values),
- `y`  — the inverse absolute temperature.

(`polyfit` will form the $(\ln R)^2$ term internally, so you don't need a separate column for it.)

👉 **Your task:** complete the three expressions.

In [ ]:
# x : natural log of the resistance
x =                      # <-- replace with your expression

# absolute temperature in kelvin: Celsius value + 273.15
T_K =                    # <-- replace with your expression

# y : inverse absolute temperature
y =                      # <-- replace with your expression

print("x = ln(R):", np.round(x, 3))
print("y = 1/T :", np.round(y, 6))

## Step 2 — Plot the data

Plot $1/T$ (vertical axis) against $\ln(R/\Omega)$ (horizontal axis) as open markers with no
connecting line, so you can see the gentle curvature that makes a straight line inadequate.

👉 **Your task:** supply the `ax.plot(...)` call (recall `'o'` with `mfc='none'` gives open circles).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# Plot x (horizontal) against y (vertical) as open circles, no connecting line.
ax.plot(                   )   # <-- pass x, y, and the format string

ax.set_xlabel(r'$\ln(R/\Omega)$')
ax.set_ylabel(r'$1/T$ (K$^{-1}$)')
ax.tick_params(direction='in')
plt.show()

## Step 3 — Quadratic (multiple) regression with uncertainties

Now fit a **degree-2** polynomial and ask for the covariance matrix. Remember the returned
coefficients are ordered highest power first, `[m2, m1, b]`, and the standard deviations are
the square roots of the diagonal of the covariance matrix.

👉 **Your task:** run the quadratic fit and pull out the three standard deviations.

In [ ]:
# Fit a degree-2 polynomial of y on x, returning the covariance matrix.
coeffs, cov =                       # <-- call np.polyfit(..., 2, cov=True)

# coeffs is ordered [m2, m1, b]
m2, m1, b = coeffs

# standard deviations = square roots of the diagonal of the covariance matrix
S_m2, S_m1, S_b =                   # <-- take sqrt of the diagonal of cov

print(f"b  = {b:.4e}  (S_b  = {S_b:.1e})")
print(f"m1 = {m1:.4e}  (S_m1 = {S_m1:.1e})")
print(f"m2 = {m2:.4e}  (S_m2 = {S_m2:.1e})")

## Step 4 — Overlay the regression curve

Use Eq. (3) with your fitted `b`, `m1`, `m2` to predict $1/T$ at each $\ln R$, then plot it as a
solid curve on top of the data.

👉 **Your task:** build the predicted array from the three parameters.

In [ ]:
# Predicted 1/T from Eq. (3):  b  +  m1 * x  +  m2 * x**2
y_calc =                   # <-- replace with your expression

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y, 'o', mfc='none', ms=6, label='data')
ax.plot(x, y_calc, '-', lw=1, label='regression curve')
ax.set_xlabel(r'$\ln(R/\Omega)$')
ax.set_ylabel(r'$1/T$ (K$^{-1}$)')
ax.tick_params(direction='in')
ax.legend()
plt.show()

## Step 5 — Report the parameters with correct significant figures

Same two rules as Exercise A: an uncertainty keeps one significant figure (two if it starts
with 1 or 2), and its last digit sets the last reported digit of the value. The `report` helper
below applies them; the units here are $\mathrm{K}^{-1}$ for $b$, $\mathrm{K}^{-1}$ for $m_1$
(per unit of $\ln R$, which is dimensionless), and likewise for $m_2$.

👉 **Your task:** call `report` for each of the three parameters.

In [ ]:
from math import log10, floor

# Format 'value +/- uncertainty unit' following the lab's two sig-fig rules.
def report(value, unc, unit=""):
    if unc == 0:
        return f"{value} {unit}"
    exp = floor(log10(abs(unc)))
    lead = int(abs(unc) / 10**exp)
    sig = 2 if lead in (1, 2) else 1
    dp = -(exp - (sig - 1))
    if dp >= 0:
        return f"{value:.{dp}f} ± {unc:.{dp}f} {unit}"
    f = 10**(-dp)
    return f"{round(value/f)*f:g} ± {round(unc/f)*f:g} {unit}"

print("b  =", report(            ))   # <-- pass b, S_b, "K^-1"
print("m1 =", report(            ))   # <-- pass m1, S_m1, "K^-1"
print("m2 =", report(            ))   # <-- pass m2, S_m2, "K^-1"


## Discussion

Answer briefly below (as Markdown):

1. Why is a straight line inadequate here? What does the sign and size of $m_2$ tell you about the curvature?
2. Using your fitted parameters, predict the temperature when $R = 3.0\ \mathrm{k\Omega}$.
   (Compute $1/T$ from Eq. (3), then invert and convert to °C.)
3. Which term contributes most to $1/T$ over the measured range? How might you decide whether a
   *cubic* term is warranted?

*Your answers here.*